In [ ]:
from pathlib import Path
import random
import pickle
from tqdm.auto import tqdm

import pandas as pd
import numpy as np
from scipy.stats import ks_2samp, chi2_contingency, ttest_ind
import rpy2.robjects.numpy2ri
from rpy2.robjects.packages import importr

rpy2.robjects.numpy2ri.activate()

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(42)

In [ ]:
real = pd.read_csv(
    Path.home()
    / "Research"
    / "Virtual-ALS-Patients"
    / "data"
    / "no_labels_no_birth_year"
    / "train.csv"
)
real.head()

In [ ]:
# filter out patients with no temporal data (all unknowns)

dfs = []

for _, pdf in real.groupby(by="REF"):
    if (pdf[[f"P{j}" for j in range(1, 13)]] == -1.0).all().all():
        continue
    dfs.append(pdf)

real = pd.concat(dfs, ignore_index=True)

In [ ]:
version = "2025-03-28 15:06:33"
fname = f"synthetic_dfs_n_samples=100_patientflow_{version}_seed=42.pkl"

with open(fname, "rb") as f:
    gen_dfs, categorical_feats_info = pickle.load(f)

In [ ]:
numerical_feats = ["Age_onset", "Height (m)", "Weight"]

categorical_feats = [
    "Gender",
    "Ethnicity",
    "NIV",
    "Tracheostomy",
    "PEG",
    "UMNvsLMN",
    "Onset",
    "Limb_O",
    "Limbs_Impairment",
    "Limbs_Side",
    "Weightloss_>10%",
    "ALS_familiar_history",
    "Ever_smoked",
    "Blood_hypertension",
    "Diabetes",
    "Dyslipidemia",
    "Thyroid",
    "Autoimmune",
    "SOD1 Mutation",
    "Stroke",
    "Cardiac_disease",
    "Primary_cancer",
    "C9orf72",
    "TARDBP mutation",
    "FUS mutation",
]

In [ ]:
p_threshold = 0.01

In [ ]:
stats = importr("stats")

In [ ]:
def generate_statistical_tests_table(real, gen_dfs, numerical_feats, categorical_feats, p_threshold=0.01):
    """
    Generate a LaTeX longtable string with statistical test results.
    
    Args:
        real: DataFrame with real data
        gen_dfs: List of generated DataFrames
        numerical_feats: List of numerical feature names
        categorical_feats: List of categorical feature names
        p_threshold: P-value threshold (default: 0.01)
    
    Returns:
        str: LaTeX longtable string
    """
    
    # Initialize result dictionaries
    ks_p_values = {}
    t_test_p_values = {}
    sigma_rule_tests = {}
    chi_square_p_values = {}
    fisher_p_tests = {}
    
    # KS-Tests for numerical features
    print("Running KS-Tests...")
    for feat in numerical_feats:
        if feat not in ks_p_values:
            ks_p_values[feat] = []

        for gen_df in gen_dfs:
            df = pd.DataFrame(
                {
                    feat.lower(): list(
                        map(lambda x: x[1][feat].iloc[0], real.groupby("REF"))
                    ),
                    "type": "real",
                }
            )
            df = pd.concat(
                [
                    df,
                    pd.DataFrame(
                        {
                            feat.lower(): list(
                                map(lambda x: x[1][feat].iloc[0], gen_df.groupby("REF"))
                            ),
                            "type": "fake",
                        }
                    ),
                ],
                ignore_index=True,
            )
            ks = ks_2samp(
                df[df["type"] == "real"][feat.lower()],
                df[df["type"] == "fake"][feat.lower()],
            )
            ks_p_values[feat].append(ks.pvalue)

    # Welch's T-Tests for numerical features
    print("Running Welch's T-Tests...")
    for feat in numerical_feats:
        if feat not in t_test_p_values:
            t_test_p_values[feat] = []

        for gen_df in gen_dfs:
            df = pd.DataFrame(
                {
                    feat.lower(): list(
                        map(lambda x: x[1][feat].iloc[0], real.groupby("REF"))
                    ),
                    "type": "real",
                }
            )
            df = pd.concat(
                [
                    df,
                    pd.DataFrame(
                        {
                            feat.lower(): list(
                                map(lambda x: x[1][feat].iloc[0], gen_df.groupby("REF"))
                            ),
                            "type": "fake",
                        }
                    ),
                ],
                ignore_index=True,
            )
            t_unequal = ttest_ind(
                df[df["type"] == "real"][feat.lower()],
                df[df["type"] == "fake"][feat.lower()],
                equal_var=False,
            )
            t_test_p_values[feat].append(t_unequal.pvalue)

    # Three Sigma Rule tests for numerical features
    print("Running Three Sigma Rule Tests...")
    sigma = 3
    for feat in numerical_feats:
        if feat not in sigma_rule_tests:
            sigma_rule_tests[feat] = []

        df = pd.DataFrame(
            {
                feat.lower(): list(map(lambda x: x[1][feat].iloc[0], real.groupby("REF"))),
                "type": "real",
            }
        )
        mean = df[df["type"] == "real"][feat.lower()].mean()
        std = df[df["type"] == "real"][feat.lower()].std()
        min_value = mean - sigma * std
        max_value = mean + sigma * std

        for gen_df in gen_dfs:
            df = pd.DataFrame(
                {
                    feat.lower(): list(
                        map(lambda x: x[1][feat].iloc[0], gen_df.groupby("REF"))
                    ),
                    "type": "fake",
                }
            )
            sigma_rule_tests[feat].append(
                not (
                    (df[feat.lower()] < min_value).any()
                    and (df[feat.lower()] > max_value).any()
                )
            )

    # Chi-Square Tests for categorical features
    print("Running Chi-Square Tests...")
    for feat in categorical_feats:
        if feat not in chi_square_p_values:
            chi_square_p_values[feat] = []

        for gen_df in gen_dfs:
            df = pd.DataFrame(
                {
                    feat.lower(): list(
                        map(lambda x: x[1][feat].iloc[0], real.groupby("REF"))
                    ),
                    "type": "real",
                }
            )
            df = pd.concat(
                [
                    df,
                    pd.DataFrame(
                        {
                            feat.lower(): list(
                                map(lambda x: x[1][feat].iloc[0], gen_df.groupby("REF"))
                            ),
                            "type": "fake",
                        }
                    ),
                ],
                ignore_index=True,
            )
            df[feat.lower()] = df[feat.lower()].apply(
                lambda x: x.lower() if isinstance(x, str) else x
            )

            counts_real = df[df["type"] == "real"][feat.lower()].value_counts().to_dict()
            counts_fake = df[df["type"] == "fake"][feat.lower()].value_counts().to_dict()
            table = np.zeros((2, len(counts_real.keys())), dtype=np.int32)

            for i, key in enumerate(counts_real.keys()):
                table[0, i] = counts_real[key]
                table[1, i] = counts_fake[key] if key in counts_fake else 0

            g, p, dof, expctd = chi2_contingency(table, lambda_="log-likelihood")
            chi_square_p_values[feat].append(p)

    # Fisher's Exact Tests for categorical features
    print("Running Fisher's Exact Tests...")
    for feat in tqdm(categorical_feats):
        if feat not in fisher_p_tests:
            fisher_p_tests[feat] = []

        for gen_df in gen_dfs:
            df = pd.DataFrame(
                {
                    feat.lower(): list(
                        map(lambda x: x[1][feat].iloc[0], real.groupby("REF"))
                    ),
                    "type": "real",
                }
            )
            df = pd.concat(
                [
                    df,
                    pd.DataFrame(
                        {
                            feat.lower(): list(
                                map(lambda x: x[1][feat].iloc[0], gen_df.groupby("REF"))
                            ),
                            "type": "fake",
                        }
                    ),
                ],
                ignore_index=True,
            )
            df[feat.lower()] = df[feat.lower()].apply(
                lambda x: x.lower() if isinstance(x, str) else x
            )

            counts_real = df[df["type"] == "real"][feat.lower()].value_counts().to_dict()
            counts_fake = df[df["type"] == "fake"][feat.lower()].value_counts().to_dict()
            table = np.zeros((2, len(counts_real.keys())), dtype=np.int32)

            for i, key in enumerate(counts_real.keys()):
                table[0, i] = counts_real[key]
                table[1, i] = counts_fake[key] if key in counts_fake else 0

            p = stats.fisher_test(table, workspace=2e8)[0][0]
            fisher_p_tests[feat].append(p)

    # Helper function to format test results
    def format_test_result(p_values, p_threshold):
        if len(p_values) == 0:
            return "\\makecell[c]{N/A}", False
        
        p_values_array = np.array(p_values)
        passing_count = (p_values_array > p_threshold).sum()
        total_count = len(p_values_array)
        
        if passing_count == 0:
            return f"\\makecell[c]{{{passing_count} / {total_count}\\\\(\\textless{{{p_threshold:.2f}}})}}"
        else:
            mean_p = p_values_array.mean()
            std_p = p_values_array.std()
            first_line = f"{passing_count} / {total_count}"
            second_line = f"({mean_p:.2f} $\\pm$ {std_p:.2f})"
            return f"\\makecell[c]{{\\textbf{{{first_line}}}\\\\\\textbf{{{second_line}}}}}"

    def format_sigma_result(sigma_tests):
        if len(sigma_tests) == 0:
            return "N/A"
        
        passing_count = sigma_tests.count(True)
        total_count = len(sigma_tests)
        result = f"{passing_count} / {total_count}"
        if passing_count > 0:
            return f"\\textbf{{{result}}}"
        else:
            return result

    def escape_latex_feature_name(name):
        """Escape special LaTeX characters in feature names"""
        # Replace underscores and other special characters
        name = name.replace('_', '\\_')
        name = name.replace('>', '\\textgreater{}')
        name = name.replace('<', '\\textless{}')
        name = name.replace('%', '\\%')
        name = name.replace('&', '\\&')
        name = name.replace('#', '\\#')
        return name

    # Generate LaTeX table
    latex_lines = []
    latex_lines.append("{\\footnotesize")
    latex_lines.append("\\begin{longtable}{ccccc}")
    latex_lines.append("\\centering")
    latex_lines.append("  & & \\makecell[c]{KS\\\\Test} &")
    latex_lines.append("  T-Test &")
    latex_lines.append("  3 Sigma \\\\")
    latex_lines.append("  \\cmidrule{3-5}")
    
    # Numerical features section
    num_numerical = len(numerical_feats)
    latex_lines.append(f"  \\multirow{{{num_numerical}}}{{*}}{{\\rotatebox[]{{90}}{{\\makecell[c]{{Numerical\\\\Features}}}}}} &")
    
    for i, feat in enumerate(numerical_feats):
        escaped_feat = escape_latex_feature_name(feat)
        
        # KS Test
        ks_cell = format_test_result(ks_p_values.get(feat, []), p_threshold)
        
        # T-Test
        t_cell = format_test_result(t_test_p_values.get(feat, []), p_threshold)
        
        # 3 Sigma
        sigma_cell = format_sigma_result(sigma_rule_tests.get(feat, []))
        
        if i == 0:
            latex_lines.append(f"  \\texttt{{{escaped_feat}}} &")
        else:
            latex_lines.append(" &")
            latex_lines.append(f"  \\texttt{{{escaped_feat}}} &")
        
        latex_lines.append(f"  {ks_cell} &")
        latex_lines.append(f"  {t_cell} &")
        
        if i == num_numerical - 1:
            latex_lines.append(f"  {sigma_cell} \\\\* \\midrule")
        else:
            latex_lines.append(f"  {sigma_cell} \\\\")

    # Categorical features section
    num_categorical = len(categorical_feats)
    latex_lines.append(f"\\multirow{{{num_categorical}}}{{*}}{{\\rotatebox[]{{90}}{{Nominal Categorical Features}}}} & & Chi-Square &")
    latex_lines.append("\\makecell[c]{Fisher's\\\\Exact Test} & \\\\")
    latex_lines.append("\\cmidrule{3-4}")
    
    for i, feat in enumerate(categorical_feats):
        escaped_feat = escape_latex_feature_name(feat)
        
        # Chi-Square Test
        chi_cell = format_test_result(chi_square_p_values.get(feat, []), p_threshold)
        
        # Fisher's Exact Test
        fisher_cell = format_test_result(fisher_p_tests.get(feat, []), p_threshold)
        
        latex_lines.append("& ")
        latex_lines.append(f"  \\texttt{{{escaped_feat}}} &")
        latex_lines.append(f"  {chi_cell} &")
        
        if i == num_categorical - 1:
            latex_lines.append(f"  {fisher_cell} & \\\\")
        else:
            latex_lines.append(f"  {fisher_cell} & \\\\")
    
    # Table caption and end
    latex_lines.append("  \\caption{}")
    latex_lines.append("  \\label{}")
    latex_lines.append("\\end{longtable}")
    latex_lines.append("}")
    
    return "\n".join(latex_lines)

In [ ]:
print(generate_statistical_tests_table(real, gen_dfs, numerical_feats, categorical_feats))